# Práctica 3: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [NLTK](https://www.nltk.org) de Python.

### Ejercicio 1

El objetivo de este ejercicio es entrenar y evaluar el rendimiento de un filtro de correo electrónico no deseado. Para ello se usará el corpus Enron-Spam, pero no se proporcionará un vocabulario fijo, sino que este deberá aprenderse a partir de los mensajes de entrenamiento. Con el objetivo de homogeneizar el vocabulario aprendido y de mejorar el rendimiento del filtro construido, se pedirá que se apliquen distintas técnicas de preprocesado.

En todos los apartados de este ejercicio se deberá realizar lo siguiente:

* Construir el filtro como una tubería de scikit-learn que concatene un vectorizador tf-idf y un modelo $k$NN clasificador con 5 vecinos y que use la métrica del coseno.
* Definir una función `procesa_mensaje` que, dado el contenido en bruto de un mensaje, aplique todos los pasos de procesamiento pedidos hasta obtener la lista de tókenes correspondiente. Esta función se deberá proporcionar como argumento `analyzer` del vectorizador tf-idf.
* Entrenar el filtro con el corpus de entrenamiento.
* Calcular la sensibilidad del filtro sobre el corpus de prueba.

In [1]:
from email import parser
from email import policy

In [2]:
analizador_mensaje = parser.Parser(policy=policy.default)

In [3]:
from pathlib import Path

In [4]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

#### Apartado 0

En este apartado se pide procesar los mensajes realizando los siguientes 3 pasos:

* Extraer el contenido de texto de los mensajes en formato HTML. Para ello hacer uso de la biblioteca [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).
* Dividir el contenido de los mensajes en secuencias de tókenes mediante el tokenizador de NLTK.
* Eliminar de los tókenes los caracteres no alfanuméricos (y eliminar por completo aquellos tókenes que no contengan caracteres alfanuméricos).

In [5]:
contenidos_mensajes_entrenamiento[-22]

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [6]:
from bs4 import BeautifulSoup

In [7]:
def elimina_html(contenido):
    return BeautifulSoup(contenido).get_text()

In [8]:
elimina_html(contenidos_mensajes_entrenamiento[-22])

'"I just wanted to write and thank you for Spur-M. \nI suffered from poor sperm count and motility. I found \nyour site and ordered Spur-M Fertility Blend for Men. \nI have wondered for years what caused low semen and sperm \ncount, and how I could improve my fertility and help my wife\nconceive. Spur-M seems to have done just that! Thank you\nfor your support."\nAndrew H., London, UK\n\n"Spur-M really does help improve fertility and effectiveness\nof sperm and semen motility. I used it for the past few months,\nand not only does it work - I also feel better to. I have \nmore energy. This is an excellent counter to low sperm count\nand motility. I\'ll be buying more!!!"\nFranz K., Bonn, Germany\n\n"I had been wondering on the causes of low semen and \nsperm count, I was searching for this type of information \nwhen I found your site. I hadn\'t been made aware of this \nproduct before then, so was quite surprised to be able \nto find a Male fertility product. Usually everything is \ngea

In [9]:
import os

os.environ['NLTK_DATA'] = '.'

In [10]:
from nltk import download

download('punkt_tab')

[nltk_data] Downloading package punkt_tab to ....
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [11]:
from nltk.tokenize import word_tokenize

In [12]:
from pprint import pprint

In [13]:
pprint(word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-22])),
       compact=True)

['``', 'I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for',
 'Spur-M', '.', 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and',
 'motility', '.', 'I', 'found', 'your', 'site', 'and', 'ordered', 'Spur-M',
 'Fertility', 'Blend', 'for', 'Men', '.', 'I', 'have', 'wondered', 'for',
 'years', 'what', 'caused', 'low', 'semen', 'and', 'sperm', 'count', ',', 'and',
 'how', 'I', 'could', 'improve', 'my', 'fertility', 'and', 'help', 'my', 'wife',
 'conceive', '.', 'Spur-M', 'seems', 'to', 'have', 'done', 'just', 'that', '!',
 'Thank', 'you', 'for', 'your', 'support', '.', "''", 'Andrew', 'H.', ',',
 'London', ',', 'UK', "''", 'Spur-M', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 '.', 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', ',', 'and',
 'not', 'only', 'does', 'it', 'work', '-', 'I', 'also', 'feel', 'better', 'to',
 '.', 'I', 'have', 'more', 'energy', '.', 'This', 'is', 'an', 'excellent

La eliminación de los caracteres no alfanuméricos se puede realizar mediante expresiones regulares, usando para ello el paquete [re](https://docs.python.org/es/3/library/re.html) de la biblioteca estándar de Python.

In [14]:
import re

In [15]:
def elimina_no_alfanumerico(contenido):
    return [re.sub(r'[^\w]', '', palabra)
            for palabra in contenido
            if re.search(r'\w', palabra)]

In [16]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [17]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-22]),
       compact=True)

['I', 'just', 'wanted', 'to', 'write', 'and', 'thank', 'you', 'for', 'SpurM',
 'I', 'suffered', 'from', 'poor', 'sperm', 'count', 'and', 'motility', 'I',
 'found', 'your', 'site', 'and', 'ordered', 'SpurM', 'Fertility', 'Blend',
 'for', 'Men', 'I', 'have', 'wondered', 'for', 'years', 'what', 'caused', 'low',
 'semen', 'and', 'sperm', 'count', 'and', 'how', 'I', 'could', 'improve', 'my',
 'fertility', 'and', 'help', 'my', 'wife', 'conceive', 'SpurM', 'seems', 'to',
 'have', 'done', 'just', 'that', 'Thank', 'you', 'for', 'your', 'support',
 'Andrew', 'H', 'London', 'UK', 'SpurM', 'really', 'does', 'help', 'improve',
 'fertility', 'and', 'effectiveness', 'of', 'sperm', 'and', 'semen', 'motility',
 'I', 'used', 'it', 'for', 'the', 'past', 'few', 'months', 'and', 'not', 'only',
 'does', 'it', 'work', 'I', 'also', 'feel', 'better', 'to', 'I', 'have', 'more',
 'energy', 'This', 'is', 'an', 'excellent', 'counter', 'to', 'low', 'sperm',
 'count', 'and', 'motility', 'I', 'll', 'be', 'buying', 'm

Debido a la naturaleza de los mensajes no deseados, algunos de ellos pueden confundir a la biblioteca Beautiful Soup, avisando esta de que el mensaje puede tratarse de una URL o de una ruta a un fichero, en lugar de un mensaje de correo electrónico. El código de la siguiente celda filtra ese tipo de avisos.

In [18]:
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

Estamos ya en condiciones de poder construir el filtro de correo electrónico no deseado.

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

In [20]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [21]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vectorizador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [22]:
from sklearn.metrics import recall_score

In [23]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9390180878552972

#### Apartado 1

En este apartado se pide incorporar al procesado de mensajes los siguientes 2 pasos:

* Expandir las contracciones típicas del idioma inglés. Usar para ello el paquete [contractions](https://github.com/kootenpv/contractions).
* Convertir todos los caracteres a minúsculas.

In [26]:
import contractions

def expande_contracciones_seguro(texto):
    # En algunos correos corruptos, contractions puede lanzar IndexError.
    # Si ocurre, devolvemos el texto original para no romper el pipeline.
    try:
        return contractions.fix(texto)
    except IndexError:
        return texto

def procesa_mensaje(contenido):
    # 1) Extraemos texto útil del posible HTML.
    contenido = elimina_html(contenido)

    # 2) Convertimos a texto por seguridad y expandimos contracciones.
    contenido = expande_contracciones_seguro(str(contenido))

    # 3) Normalizamos a minúsculas.
    contenido = contenido.lower()

    # 4) Tokenizamos el texto ya normalizado.
    contenido = word_tokenize(contenido)

    # 5) Eliminamos caracteres no alfanuméricos y tokens vacíos.
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

# Mantenemos exactamente el mismo clasificador pedido (TF-IDF + kNN coseno).
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9441860465116279

#### Apartado 2

Palabras vacías (_stop words_, en inglés) es el nombre que reciben las palabras tales como artículos, pronombres y preposiciones que se considera que no aportan significado para un sistema de procesamiento del lenguaje natural y que, por tanto, deben eliminarse durante las operaciones de preprocesado de texto. El conjunto adecuado de palabras vacías a usar depende del sistema concreto que se esté construyendo, e incluso puede resultar conveniente no hacer uso de esta técnica.

NLTK provee de conjuntos genéricos de palabras vacías para distintos idiomas.

In [27]:
download('stopwords')

[nltk_data] Downloading package stopwords to ....
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [28]:
from nltk.corpus import stopwords

In [29]:
palabras_vacias_ingles = stopwords.words('english')
pprint(palabras_vacias_ingles, compact=True)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an',
 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been',
 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn',
 "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't",
 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from',
 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven',
 "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself',
 "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in',
 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself',
 "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most',
 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not',
 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours',
 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "s

En este apartado se pide incorporar al procesado de mensajes la eliminación de palabras vacías.

In [30]:
# Pasamos la lista de stopwords a conjunto para búsquedas rápidas.
palabras_vacias_ingles_set = set(palabras_vacias_ingles)

def procesa_mensaje(contenido):
    # 1) Extraemos texto útil del posible HTML.
    contenido = elimina_html(contenido)

    # 2) Expandimos contracciones de forma robusta.
    contenido = expande_contracciones_seguro(str(contenido))

    # 3) Normalizamos a minúsculas.
    contenido = contenido.lower()

    # 4) Tokenizamos.
    contenido = word_tokenize(contenido)

    # 5) Eliminamos símbolos no alfanuméricos y tokens vacíos.
    contenido = elimina_no_alfanumerico(contenido)

    # 6) Eliminamos palabras vacías (stop words).
    contenido = [
        token for token in contenido
        if token not in palabras_vacias_ingles_set
    ]
    return contenido

# Mantenemos el clasificador pedido: TF-IDF + kNN (k=5, distancia coseno).
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

# Entrenamos con entrenamiento y evaluamos sensibilidad (recall) en prueba.
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

0.9483204134366925

#### Apartado 3

Por razones gramaticales, en un documento de texto van a aparecer con seguridad diferentes formas de una palabra, como organizar, organiza y organizando. Además, existen familias de palabras relacionadas derivativamente con significados similares, como democracia, democrático y democratización. En muchas situaciones, parece que sería útil reducir esos conjuntos de palabras a una raíz común. Para ello se suelen usar los procedimientos de _stemming_ y lematización.

_Stemming_ generalmente se refiere a un proceso heurístico rudimentario que corta los extremos de las palabras con la esperanza de lograr el objetivo correctamente la mayor parte del tiempo y, a menudo, incluye la eliminación de afijos derivativos. La lematización generalmente se refiere a hacer las cosas correctamente con el uso de un vocabulario y análisis morfológico de las palabras, normalmente con el objetivo de eliminar únicamente las terminaciones flexivas y devolver la forma base o de diccionario de una palabra, lo que se conoce como lema.

NLTK provee de varios algoritmos de _stemming_ y lematización. En este apartado se pide incorporar al procesado de mensajes el procedimiento de _stemming_ mediante el [algoritmo de Lancaster](https://www.nltk.org/api/nltk.stem.lancaster.html).

In [31]:
from nltk.stem.lancaster import LancasterStemmer

In [ ]:
# Creamos el stemmer de Lancaster para reducir cada palabra a su raíz.
stemmer_lancaster = LancasterStemmer()

def procesa_mensaje(contenido):
    # 1) Extraemos texto útil del posible HTML.
    contenido = elimina_html(contenido)

    # 2) Expandimos contracciones de forma robusta.
    contenido = expande_contracciones_seguro(str(contenido))

    # 3) Normalizamos a minúsculas.
    contenido = contenido.lower()

    # 4) Tokenizamos.
    contenido = word_tokenize(contenido)

    # 5) Eliminamos símbolos no alfanuméricos y tokens vacíos.
    contenido = elimina_no_alfanumerico(contenido)

    # 6) Eliminamos palabras vacías.
    contenido = [
        token for token in contenido
        if token not in palabras_vacias_ingles_set
    ]

    # 7) Aplicamos stemming de Lancaster.
    contenido = [stemmer_lancaster.stem(token) for token in contenido]
    return contenido

# Mantenemos el clasificador pedido: TF-IDF + kNN (k=5, distancia coseno).
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

# Entrenamos y evaluamos sensibilidad (recall) en prueba.
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

### Ejercicio 2

En el cuaderno NLTK.ipynb se ha construido un sistema de predicción de texto en español basado en modelos de $n$-gramas. Estos modelos se han entrenado a partir de un corpus de textos en español que se ha usado en bruto. El objetivo de este ejercicio es recrear la construcción del sistema de predicción de texto, pero usando una versión normalizada del corpus.

#### Apartado 1

En este apartado se pide:

1. Leer el corpus guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt` y dividirlo en un corpus de entrenamiento y un corpus de prueba.
2. Construir modelos unigramas, bigramas y trigramas, con y sin suavizado, a partir del corpus de entrenamiento normalizado convirtiendo todas las palabras a minúsculas.
3. Seleccionar el modelo con menor perplejidad sobre el corpus de prueba normalizado convirtiendo todas las palabras a minúsculas.

#### Apartado 2

Redefinir la función `predice_palabras` de tal forma que prediga, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir, actuando como sigue:

* Si todas las letras del prefijo escrito están en minúsculas, entonces debe predecir palabras en minúsculas.
* Si todas las letras del prefijo escrito están en mayúsculas, entonces debe predecir palabras en mayúsculas.
* Si el prefijo escrito mezcla letras en minúsculas y en mayúsculas, entonces:
  * Si la primera letra del prefijo está en minúsculas, entonces debe predecir palabras en minúsculas.
  * Si la primera letra del prefijo está en mayúsculas, entonces debe predecir palabras con la primera letra en mayúsculas y el resto en minúsculas.

In [ ]:
predice_palabras('nat', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['natural', 'naturalmente', 'nata', 'naturales', 'natacha']

In [ ]:
predice_palabras('NAT', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['NATURAL', 'NATURALMENTE', 'NATA', 'NATURALES', 'NATACHA']

In [ ]:
predice_palabras('nAt', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['natural', 'naturalmente', 'nata', 'naturales', 'natacha']

In [ ]:
predice_palabras('NaT', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['Natural', 'Naturalmente', 'Nata', 'Naturales', 'Natacha']